In [ ]:
!pip install scikit-learn

In [ ]:
# Importando as bibliotecas necessárias para análise e visualização
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Ignorar avisos irrelevantes
import warnings
warnings.filterwarnings("ignore")

# Carregar o arquivo CSV com os dados de qualidade da água
df = pd.read_csv('/content/water_potability.csv')

# Exibir as primeiras linhas do dataset
df.head()

In [ ]:
# Verificando o tamanho do dataset
print(f'Total de linhas: {df.shape[0]}')
print(f'Total de colunas: {df.shape[1]}')

# Verificando se há dados faltantes
df.isnull().sum()

In [ ]:
# Preencher valores ausentes com a mediana de cada coluna
df.fillna(df.median(), inplace=True)

# Verificar novamente se ainda existem valores nulos
df.isnull().sum()

In [ ]:
# Ver um resumo estatístico dos dados
df.describe()

# Matriz de correlação entre variáveis
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='viridis')
plt.title('Correlação entre variáveis da água')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separar os dados sem a coluna 'Potability', pois é um projeto não supervisionado
X = df.drop('Potability', axis=1)

# Padronizar os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


**TRATAMENTO DOS DADOS**

In [ ]:

# Remove a coluna 'Potability' e garante que só fique valores numéricos
X = df.drop('Potability', axis=1)

# Preenche qualquer valor nulo com a mediana da respectiva coluna
X = X.fillna(X.median())

# Padronização dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# CLUSTERIZAÇÃO
# Aplica o algoritmo KMeans com 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Adiciona a coluna 'Cluster' ao DataFrame original
df['Cluster'] = clusters

# Visualiza os primeiros resultados com o cluster adicionado
df.head()

In [ ]:
# Ver a distribuição dos clusters
sns.countplot(x='Cluster', data=df)
plt.title('Distribuição dos Grupos de Qualidade da Água')
plt.xlabel('Grupo')
plt.ylabel('Quantidade de Regiões')
plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df['Cluster'], palette='Set2')
plt.title('Visualização dos Clusters em 2D (PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Cluster')
plt.show()


In [ ]:
# Boxplots para comparar distribuição por cluster
plt.figure(figsize=(15,10))
for i, col in enumerate(X.columns):
    plt.subplot(3, 3, i+1)
    sns.boxplot(x='Cluster', y=col, data=df)
    plt.title(f'{col} por Cluster')
    plt.tight_layout()


In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['ph'], bins=30, kde=True, color='blue')
plt.title('Distribuição do pH')
plt.xlabel('pH')
plt.ylabel('Frequência')
plt.show()

In [ ]:
df.drop(columns='Cluster').hist(figsize=(15,10), bins=30, edgecolor='black')
plt.suptitle('Distribuição das variáveis - Qualidade da Água', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Médias das variáveis por cluster
cluster_means = df.groupby('Cluster').mean()

plt.figure(figsize=(12,6))
sns.heatmap(cluster_means.T, annot=True, cmap='YlGnBu')
plt.title('Médias das Variáveis por Cluster')
plt.xlabel('Cluster')
plt.ylabel('Variável')
plt.tight_layout()
plt.show()


In [ ]:
# Amostra para visualização rápida
sample_df = df.sample(300, random_state=42)

sns.pairplot(sample_df, vars=['ph', 'Hardness', 'Solids', 'Conductivity'], hue='Cluster', palette='Set2')
plt.suptitle('Relação entre Variáveis - Amostra de 300', y=1.02)
plt.show()


In [ ]:
print(df['Cluster'].value_counts())


In [ ]:
# Conta quantas regiões tem em cada cluster
cluster_counts = df['Cluster'].value_counts().sort_index()

# Define os nomes dos grupos (você pode alterar)
labels = ['Baixo Risco', 'Médio Risco', 'Alto Risco']

# Cores personalizadas
colors = ['#66c2a5', '#fc8d62', '#8da0cb']

# Gráfico de pizza
plt.figure(figsize=(6,6))
plt.pie(cluster_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Distribuição de Regiões por Nível de Risco')
plt.axis('equal')  # Deixa o gráfico redondo
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='Organic_carbon', hue='Cluster', fill=True)
plt.title('Densidade de Carbono Orgânico por Cluster')
plt.xlabel('Carbono Orgânico')
plt.ylabel('Densidade Estimada')
plt.show()


In [ ]:
sns.lmplot(data=df, x='Hardness', y='Solids', hue='Cluster', height=6, aspect=1.2)
plt.title('Relação entre Dureza e Sólidos com Linha de Tendência')
plt.xlabel('Dureza')
plt.ylabel('Sólidos')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='Cluster', y='Sulfate', ci="sd", palette='Set3')
plt.title('Média de Sulfato por Cluster com Desvio Padrão')
plt.xlabel('Cluster')
plt.ylabel('Concentração de Sulfato')
plt.show()


In [ ]:
# Tabela com as médias das variáveis por cluster
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary


In [ ]:
# Cálculo da inércia para diferentes valores de k
inertia = []
for k in range(1, 10):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Plot do gráfico de cotovelo
plt.figure(figsize=(8,5))
plt.plot(range(1, 10), inertia, marker='o')
plt.title('Método do Cotovelo - Escolha do Número de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('Inércia')
plt.xticks(range(1, 10))
plt.grid(True)
plt.show()
